# UCS645 Lab 8 - Colab Runner\n
This notebook helps you run Assignment 8 CUDA exercises in Google Colab.\n
\n
Workflow:\n
1. Upload `Assignment8.zip` (containing all `.cu` files and `Makefile`).\n
2. Compile exercises with detected GPU architecture.\n
3. Run Exercise 1 to 4.\n
4. Optionally download MNIST and run Exercise 5.\n

In [43]:
!nvidia-smi\n
!nvcc --version\n

/bin/bash: line 1: nvidia-smin: command not found
nvcc fatal   : Unknown option '--versionn'


## Step 1: Upload Assignment Folder as Zip\n
Zip your local `Lab8/Assignment8` folder and upload it below as `Assignment8.zip`.\n

In [44]:
from google.colab import files
uploaded = files.upload()

Saving Assignment8.zip to Assignment8 (1).zip


In [45]:
!rm -rf /content/Assignment8
!mkdir -p /content/Assignment8
!unzip -o Assignment8.zip -d /content/Assignment8
!ls -lah /content/Assignment8

Archive:  Assignment8.zip
  inflating: /content/Assignment8/ex01_cuda_basics.cu  
  inflating: /content/Assignment8/ex02_memory_hierarchy.cu  
  inflating: /content/Assignment8/ex03_ml_primitives.cu  
  inflating: /content/Assignment8/ex04_cnn_layers.cu  
  inflating: /content/Assignment8/ex05_mnist_cnn.cu  
  inflating: /content/Assignment8/Makefile  
total 140K
drwxr-xr-x 2 root root 4.0K Apr 27 07:25 .
drwxr-xr-x 1 root root 4.0K Apr 27 07:25 ..
-rw-rw-rw- 1 root root  20K Apr 20 01:50 ex01_cuda_basics.cu
-rw-rw-rw- 1 root root  21K Apr 20 01:50 ex02_memory_hierarchy.cu
-rw-rw-rw- 1 root root  23K Apr 20 01:50 ex03_ml_primitives.cu
-rw-rw-rw- 1 root root  25K Apr 20 01:50 ex04_cnn_layers.cu
-rw-rw-rw- 1 root root  32K Apr 20 08:10 ex05_mnist_cnn.cu
-rw-rw-rw- 1 root root 1.6K Apr 20 01:50 Makefile


## Step 2: Compile Exercise 1 to 4\n

In [46]:
%%bash
set -e
cd /content/Assignment8

# Colab-safe build flags: avoid brittle arch auto-detection
NVCC_FLAGS="-O2"
echo "Using NVCC flags: ${NVCC_FLAGS}"

nvcc ${NVCC_FLAGS} ex01_cuda_basics.cu -o ex01_cuda_basics
nvcc ${NVCC_FLAGS} ex02_memory_hierarchy.cu -o ex02_memory_hierarchy
nvcc ${NVCC_FLAGS} ex03_ml_primitives.cu -o ex03_ml_primitives -lm
nvcc ${NVCC_FLAGS} ex04_cnn_layers.cu -o ex04_cnn_layers -lcublas

echo "Build done for ex01-ex04"

Using NVCC flags: -O2
Build done for ex01-ex04


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex01_cuda_basics.cu(305): warning #177-D: variable "elapsed_ms" was declared but never referenced
          float elapsed_ms = 0.0f;
                ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
ex02_memory_hierarchy.cu(140): warning #177-D: variable "tile" was declared but never referenced
      __attribute__((shared)) float tile[256];
                                    ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex02_memory_hierarchy.cu(141): warning #177-D: variable "i" was declared but never referenced
      int i = blockIdx.x * blockDim.x +

In [47]:
%%bash
set -e
cd /content/Assignment8
mkdir -p logs

echo 'Running ex01...'
./ex01_cuda_basics | tee logs/ex01.log

echo 'Running ex02...'
./ex02_memory_hierarchy | tee logs/ex02.log

echo 'Running ex03...'
./ex03_ml_primitives | tee logs/ex03.log

echo 'Running ex04...'
./ex04_cnn_layers | tee logs/ex04.log

Running ex01...

  CUDA DIY Exercise 1: Basics & Memory
  GPU: Tesla T4  (SM 7.5)  VRAM: 15637 MB

[Section A] Reference:
  [A1-VectorAdd] N=1048576  CPU=2.7 ms  GPU=0.26 ms  Speedup=10.5x  [PASS]

[Section B] DIY Exercises:
  [B1-VectorScale] k=3.14  [FAIL] -- check your kernel
  [B2-SquaredDiff] [FAIL] -- check your kernel

  [B3-LaunchConfig] threads_per_block=256
           N    blocks    total_threads   covers_all?
  ---------------------------------------------------
           1         0                0        [FAIL]
         100         0                0        [FAIL]
         256         0                0        [FAIL]
         257         0                0        [FAIL]
        1024         0                0        [FAIL]
       10000         0                0        [FAIL]
     1048576         0                0        [FAIL]

  [B4-MemoryBandwidth]
   Size (MB)    H2D (GB/s)    D2H (GB/s)
  ----------------------------------------
           1           0.0          

## Inference Dashboard (C++ + Command Line)\n
The next cells avoid Python-heavy parsing.\n
They compile a C++ parser, produce CSV tables, and generate graph images using command-line tools.\n

In [48]:
%%bash
set -e
cd /content/Assignment8

apt-get update -y >/dev/null
apt-get install -y gnuplot-nox >/dev/null

cat > parse_logs.cpp << 'CPP'
#include <filesystem>
#include <fstream>
#include <iostream>
#include <regex>
#include <sstream>
#include <string>
#include <vector>
using namespace std;

static string read_all(const string& path) {
    ifstream f(path);
    if (!f.is_open()) return "";
    stringstream ss; ss << f.rdbuf();
    return ss.str();
}


static int count_token(const string& s, const string& token) {
    int c = 0; size_t pos = 0;
    while ((pos = s.find(token, pos)) != string::npos) { ++c; pos += token.size(); }
    return c;
}


static void write_text(const string& path, const string& txt) {
    ofstream f(path); f << txt;
}


int main(int argc, char** argv) {
    if (argc < 3) {
        cerr << "Usage: parse_logs <logs_dir> <out_dir>\n";
        return 1;
    }
    string logs = argv[1];
    string outd = argv[2];
    filesystem::create_directories(outd);

    vector<string> exes = {"ex01", "ex02", "ex03", "ex04", "ex05"};
    string summary = "exercise,pass_count,fail_count\n";
    for (const auto& e : exes) {
        string txt = read_all(logs + "/" + e + ".log");
        summary += e + "," + to_string(count_token(txt, "[PASS]")) + "," + to_string(count_token(txt, "[FAIL]")) + "\n";
    }
    write_text(outd + "/summary.csv", summary);

    string ex1 = read_all(logs + "/ex01.log");
    smatch m;
    regex vec_re(R"(\[A1-VectorAdd\].*CPU=([0-9.]+)\s+ms\s+GPU=([0-9.]+)\s+ms\s+Speedup=([0-9.]+)\_x)");
    string vectoradd = "CPU_ms,GPU_ms,Speedup_x\n";
    if (regex_search(ex1, m, vec_re)) {
        vectoradd += m[1].str() + "," + m[2].str() + "," + m[3].str() + "\n";
    }
    write_text(outd + "/vectoradd.csv", vectoradd);

    regex bw_re(R"(^\s*(1|8|64|256|512)\s+([0-9.]+)\s+([0-9.]+)\s*$)");
    string bandwidth = "Size_MB,H2D_GBps,D2H_GBps\n";
    {
        istringstream iss(ex1); string line; smatch mm;
        while (getline(iss, line)) {
            if (regex_match(line, mm, bw_re)) {
                bandwidth += mm[1].str() + "," + mm[2].str() + "," + mm[3].str() + "\n";
            }
        }
    }
    write_text(outd + "/bandwidth.csv", bandwidth);

    string ex2 = read_all(logs + "/ex02.log");
    regex bank_re(R"(^\s*(1|2|4|8|16|32)\s+([0-9.]+).*$)");
    string bank = "Stride,Time_us\n";
    {
        istringstream iss(ex2); string line; smatch mm;
        while (getline(iss, line)) {
            if (regex_match(line, mm, bank_re)) {
                bank += mm[1].str() + "," + mm[2].str() + "\n";
            }
        }
    }
    write_text(outd + "/bank_conflict.csv", bank);

    string ex4 = read_all(logs + "/ex04.log");
    regex gemm_re(R"(^\s*(128|256|512|1024)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s*$)");
    string gemm = "Size,Naive_ms,Tiled_ms,cuBLAS_ms,cuBLAS_GFLOPS\n";
    {
        istringstream iss(ex4); string line; smatch mm;
        while (getline(iss, line)) {
            if (regex_match(line, mm, gemm_re)) {
                gemm += mm[1].str() + "," + mm[2].str() + "," + mm[3].str() + "," + mm[4].str() + "," + mm[5].str() + "\n";
            }
        }
    }
    write_text(outd + "/gemm.csv", gemm);

    string ex5 = read_all(logs + "/ex05.log");
    regex ep_re(R"(--- Epoch\s+([0-9]+)\s+Done\s+AvgLoss=([0-9.]+))");
    string epoch = "Epoch,AvgLoss\n";
    {
        auto b = sregex_iterator(ex5.begin(), ex5.end(), ep_re);
        auto e = sregex_iterator();
        for (auto it = b; it != e; ++it) {
            epoch += (*it)[1].str() + "," + (*it)[2].str() + "\n";
        }
    }
    write_text(outd + "/epoch_loss.csv", epoch);

    cout << "CSV files written to " << outd << "\n";
    return 0;
}
CPP

g++ -O2 -std=c++17 parse_logs.cpp -o parse_logs
echo "Built parser: /content/Assignment8/parse_logs"


Built parser: /content/Assignment8/parse_logs


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [49]:
%%bash
set -e
cd /content/Assignment8

mkdir -p analysis/plots
./parse_logs logs analysis

echo "==== PASS/FAIL Summary ===="
column -s, -t analysis/summary.csv

echo "==== VectorAdd ===="
column -s, -t analysis/vectoradd.csv

echo "==== Bandwidth ===="
column -s, -t analysis/bandwidth.csv

echo "==== Bank Conflict ===="
column -s, -t analysis/bank_conflict.csv

echo "==== GEMM ===="
column -s, -t analysis/gemm.csv

gnuplot << 'GP'
set datafile separator ','
set terminal pngcairo size 1000,500
set output 'analysis/plots/pass_fail.png'
set title 'PASS vs FAIL by Exercise'
set style data histogram
set style histogram clustered gap 1
set style fill solid border -1
set boxwidth 0.9
set xtics rotate by -20
plot 'analysis/summary.csv' using 2:xtic(1) title 'PASS', '' using 3 title 'FAIL'
GP

gnuplot << 'GP'
set datafile separator ','
set terminal pngcairo size 1000,500
set output 'analysis/plots/bandwidth.png'
set title 'PCIe Bandwidth vs Transfer Size'
set xlabel 'Size (MB)'
set ylabel 'GB/s'
set logscale x 2
set xtics (1,8,64,256,512)
plot 'analysis/bandwidth.csv' using 1:2 with linespoints lw 2 title 'H2D', \
     'analysis/bandwidth.csv' using 1:3 with linespoints lw 2 title 'D2H'
GP

gnuplot << 'GP'
set datafile separator ','
set terminal pngcairo size 1000,500
set output 'analysis/plots/bank_conflict.png'
set title 'Bank Conflict Cost'
set xlabel 'Stride'
set ylabel 'Time (us)'
set logscale x 2
set xtics (1,2,4,8,16,32)
plot 'analysis/bank_conflict.csv' using 1:2 with linespoints lw 2 title 'kernel time'
GP

gnuplot << 'GP'
set datafile separator ','
set terminal pngcairo size 1000,500
set output 'analysis/plots/gemm_runtime.png'
set title 'GEMM Runtime Comparison'
set xlabel 'Matrix Size (N for NxN)'
set ylabel 'Time (ms)'
plot 'analysis/gemm.csv' using 1:2 with linespoints lw 2 title 'Naive', \
     'analysis/gemm.csv' using 1:3 with linespoints lw 2 title 'Tiled', \
     'analysis/gemm.csv' using 1:4 with linespoints lw 2 title 'cuBLAS'
GP

gnuplot << 'GP'
set datafile separator ','
set terminal pngcairo size 1000,500
set output 'analysis/plots/cublas_gflops.png'
set title 'cuBLAS Throughput'
set xlabel 'Matrix Size (N for NxN)'
set ylabel 'GFLOPS'
plot 'analysis/gemm.csv' using 1:5 with linespoints lw 2 title 'cuBLAS GFLOPS'
GP

echo "Generated files:"
ls -lah analysis
ls -lah analysis/plots


CSV files written to analysis
==== PASS/FAIL Summary ====
exercise  pass_count  fail_count
ex01      2           10
ex02      2           5
ex03      2           7
ex04      1           4
ex05      0           0
==== VectorAdd ====
CPU_ms  GPU_ms  Speedup_x
==== Bandwidth ====
Size_MB  H2D_GBps  D2H_GBps
1        0.0       0.0
8        0.0       0.0
64       0.0       0.0
256      0.0       0.0
512      0.0       0.0
==== Bank Conflict ====
Stride  Time_us
1       0.00
2       0.00
4       0.00
8       0.00
16      0.00
32      0.00
==== GEMM ====
Size  Naive_ms  Tiled_ms  cuBLAS_ms  cuBLAS_GFLOPS
128   0.02      0.00      0.00       4194304.0
256   0.08      0.00      0.00       33554432.0
512   0.53      0.00      0.00       268435456.0
1024  4.14      0.00      0.00       2147483648.0
Generated files:
total 36K
drwxr-xr-x 3 root root 4.0K Apr 27 07:26 .
drwxr-xr-x 4 root root 4.0K Apr 27 07:26 ..
-rw-r--r-- 1 root root   81 Apr 27 07:26 bandwidth.csv
-rw-r--r-- 1 root root   59 Apr 

## Step 3 (Optional): Prepare MNIST for Exercise 5\n
Exercise 5 expects binary IDX files in `/content/Assignment8/data`.\n

In [50]:
%%bash
set -e
cd /content/Assignment8
mkdir -p data
cd data

base=https://ossci-datasets.s3.amazonaws.com/mnist
wget -nc ${base}/train-images-idx3-ubyte.gz
wget -nc ${base}/train-labels-idx1-ubyte.gz
wget -nc ${base}/t10k-images-idx3-ubyte.gz
wget -nc ${base}/t10k-labels-idx1-ubyte.gz

gunzip -f *.gz
ls -lah

total 53M
drwxr-xr-x 2 root root 4.0K Apr 27 07:26 .
drwxr-xr-x 5 root root 4.0K Apr 27 07:26 ..
-rw-r--r-- 1 root root 7.5M Mar  4  2020 t10k-images-idx3-ubyte
-rw-r--r-- 1 root root 9.8K Mar  4  2020 t10k-labels-idx1-ubyte
-rw-r--r-- 1 root root  45M Mar  4  2020 train-images-idx3-ubyte
-rw-r--r-- 1 root root  59K Mar  4  2020 train-labels-idx1-ubyte


--2026-04-27 07:26:08--  https://ossci-datasets.s3.amazonaws.com/mnist/train-images-idx3-ubyte.gz
Resolving ossci-datasets.s3.amazonaws.com (ossci-datasets.s3.amazonaws.com)... 52.217.93.12, 54.231.199.25, 3.5.31.125, ...
Connecting to ossci-datasets.s3.amazonaws.com (ossci-datasets.s3.amazonaws.com)|52.217.93.12|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9912422 (9.5M) [application/x-gzip]
Saving to: ‘train-images-idx3-ubyte.gz’

     0K .......... .......... .......... .......... ..........  0% 2.61M 4s
    50K .......... .......... .......... .......... ..........  1% 2.64M 4s
   100K .......... .......... .......... .......... ..........  1% 2.65M 4s
   150K .......... .......... .......... .......... ..........  2%  127M 3s
   200K .......... .......... .......... .......... ..........  2%  276M 2s
   250K .......... .......... .......... .......... ..........  3% 2.68M 2s
   300K .......... .......... .......... .......... ..........  3%  177M 2s
  

In [51]:
%%bash
set -e
cd /content/Assignment8

GPU_CC="sm_$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -n1 | tr -d ' .')"
echo "Detected architecture: ${GPU_CC}"

nvcc -O2 -arch=${GPU_CC} ex05_mnist_cnn.cu -o ex05_mnist_cnn -lcudnn -lcublas -lm
./ex05_mnist_cnn | tee /content/Assignment8/logs/ex05.log

Detected architecture: sm_75

  CUDA DIY Exercise 5: MNIST CNN (cuDNN + cuBLAS)
  GPU: Tesla T4  Compute: 7.5  VRAM: 15637 MB

[✓] Loaded 60000 MNIST samples from data/train-images-idx3-ubyte
[✓] Loaded 10000 MNIST samples from data/t10k-images-idx3-ubyte

[Training] Starting for 10 epochs...

  Epoch 1  Batch [0/234]  AvgLoss=2.3026
  Epoch 1  Batch [50/234]  AvgLoss=2.3027
  Epoch 1  Batch [100/234]  AvgLoss=2.3018
  Epoch 1  Batch [150/234]  AvgLoss=2.3025
  Epoch 1  Batch [200/234]  AvgLoss=2.3031
  --- Epoch 1 Done  AvgLoss=2.3029 ---
  Epoch 1 complete in 0.1 s

  Epoch 2  Batch [0/234]  AvgLoss=2.3026
  Epoch 2  Batch [50/234]  AvgLoss=2.3027
  Epoch 2  Batch [100/234]  AvgLoss=2.3018
  Epoch 2  Batch [150/234]  AvgLoss=2.3025
  Epoch 2  Batch [200/234]  AvgLoss=2.3031
  --- Epoch 2 Done  AvgLoss=2.3029 ---
  Epoch 2 complete in 0.1 s

  Epoch 3  Batch [0/234]  AvgLoss=2.3026
  Epoch 3  Batch [50/234]  AvgLoss=2.3027
  Epoch 3  Batch [100/234]  AvgLoss=2.3018
  Epoch 3  Batch [1

ex05_mnist_cnn.cu(95): warning #1650-D: result of call is not used
      fread(b, 1, 4, f);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex05_mnist_cnn.cu(129): warning #1650-D: result of call is not used
          fread(buf, 1, 784, fi);
          ^

ex05_mnist_cnn.cu(132): warning #1650-D: result of call is not used
          unsigned char lbl; fread(&lbl, 1, 1, fl);
                             ^

ex05_mnist_cnn.cu(95): warning #1650-D: result of call is not used
      fread(b, 1, 4, f);
      ^

Remark: The warnings can be suppressed with "-diag-suppress <warning-number>"

ex05_mnist_cnn.cu(129): warning #1650-D: result of call is not used
          fread(buf, 1, 784, fi);
          ^

ex05_mnist_cnn.cu(132): warning #1650-D: result of call is not used
          unsigned char lbl; fread(&lbl, 1, 1, fl);
                             ^

ex05_mnist_cnn.cu(233): warning #177-D: variable "alpha" was declared but never referenced
      float alp

## Inference for Exercise 5 Training\n
This cell builds an epoch-loss table and graph from the ex05 log using command-line tools.\n

In [52]:
%%bash
set -e
cd /content/Assignment8

mkdir -p logs analysis/plots

if [ ! -x ./parse_logs ]; then
  echo "parse_logs not found. Run the parser build cell first."
  exit 0
fi

if [ -x ./ex05_mnist_cnn ]; then
  ./ex05_mnist_cnn | tee logs/ex05.log >/dev/null
else
  echo "ex05_mnist_cnn binary not present. Skipping run; parser will use existing logs if any."
fi

./parse_logs logs analysis

echo "==== Epoch Loss Table ===="
if command -v column >/dev/null 2>&1; then
  column -s, -t analysis/epoch_loss.csv
else
  cat analysis/epoch_loss.csv
fi

if [ "$(wc -l < analysis/epoch_loss.csv)" -gt 1 ]; then
  gnuplot << 'GP'
  set datafile separator ','
  set terminal pngcairo size 1000,500
  set output 'analysis/plots/epoch_loss.png'
  set title 'Exercise 5 Training Loss by Epoch'
  set xlabel 'Epoch'
  set ylabel 'Average Loss'
  plot 'analysis/epoch_loss.csv' using 1:2 with linespoints lw 2 title 'Avg Loss'
GP
  echo "Generated: analysis/plots/epoch_loss.png"
else
  echo "No epoch rows found in ex05 log."
fi

CSV files written to analysis
==== Epoch Loss Table ====
Epoch  AvgLoss
1      2.3029
2      2.3029
3      2.3029
4      2.3029
5      2.3029
6      2.3029
7      2.3029
8      2.3029
9      2.3029
10     2.3029
Generated: analysis/plots/epoch_loss.png
